The following additional libraries are needed to run this
notebook. Note that running on Colab is experimental, please report a Github
issue if you have any problem.

In [1]:
!pip install -U mxnet-cu101==1.7.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 54.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.5/846.0 MB 5.7 MB/s eta 0:01:56
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/pip/_vendor/urllib3/response.py", line 438, in _error_catcher
    yield
  File "/usr/local/lib/python3.13/dist-packages/pip/_vendor/urllib3/response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ~~~~~~~~~~~~~^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_vendor/urllib3/response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ~~~~~~~~~~~~~^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pip/_vendor/cachecontrol/filewrapper.py", line 98,

# 延后初始化
:label:`sec_deferred_init`

到目前为止，我们忽略了建立网络时需要做的以下这些事情：

* 我们定义了网络架构，但没有指定输入维度。
* 我们添加层时没有指定前一层的输出维度。
* 我们在初始化参数时，甚至没有足够的信息来确定模型应该包含多少参数。

有些读者可能会对我们的代码能运行感到惊讶。
毕竟，深度学习框架无法判断网络的输入维度是什么。
这里的诀窍是框架的*延后初始化*（defers initialization），
即直到数据第一次通过模型传递时，框架才会动态地推断出每个层的大小。

在以后，当使用卷积神经网络时，
由于输入维度（即图像的分辨率）将影响每个后续层的维数，
有了该技术将更加方便。
现在我们在编写代码时无须知道维度是什么就可以设置参数，
这种能力可以大大简化定义和修改模型的任务。
接下来，我们将更深入地研究初始化机制。

## 实例化网络

首先，让我们实例化一个多层感知机。


In [2]:
from mxnet import np, npx
from mxnet.gluon import nn

npx.set_np()

def get_net():
    net = nn.Sequential()
    net.add(nn.Dense(256, activation='relu'))
    net.add(nn.Dense(10))
    return net

net = get_net()

ModuleNotFoundError: No module named 'mxnet'

此时，因为输入维数是未知的，所以网络不可能知道输入层权重的维数。
因此，框架尚未初始化任何参数，我们通过尝试访问以下参数进行确认。


In [ ]:
print(net.collect_params)
print(net.collect_params())

注意，当参数对象存在时，每个层的输入维度为-1。
MXNet使用特殊值-1表示参数维度仍然未知。
此时，尝试访问`net[0].weight.data()`将触发运行时错误，
提示必须先初始化网络，然后才能访问参数。
现在让我们看看当我们试图通过`initialize`函数初始化参数时会发生什么。


In [ ]:
net.initialize()
net.collect_params()

如我们所见，一切都没有改变。
当输入维度未知时，调用`initialize`不会真正初始化参数。
而是会在MXNet内部声明希望初始化参数，并且可以选择初始化分布。


接下来让我们将数据通过网络，最终使框架初始化参数。


In [ ]:
X = np.random.uniform(size=(2, 20))
net(X)

net.collect_params()

一旦我们知道输入维数是20，框架可以通过代入值20来识别第一层权重矩阵的形状。
识别出第一层的形状后，框架处理第二层，依此类推，直到所有形状都已知为止。
注意，在这种情况下，只有第一层需要延迟初始化，但是框架仍是按顺序初始化的。
等到知道了所有的参数形状，框架就可以初始化参数。

## 小结

* 延后初始化使框架能够自动推断参数形状，使修改模型架构变得容易，避免了一些常见的错误。
* 我们可以通过模型传递数据，使框架最终初始化参数。

## 练习

1. 如果指定了第一层的输入尺寸，但没有指定后续层的尺寸，会发生什么？是否立即进行初始化？
1. 如果指定了不匹配的维度会发生什么？
1. 如果输入具有不同的维度，需要做什么？提示：查看参数绑定的相关内容。


[Discussions](https://discuss.d2l.ai/t/5770)


1. 指定第一层尺寸但不指定后续尺寸，是否会立即初始化？这取决于你所使用的具体深度学习框架及其工作机制：支持静态形状推断的框架（如 Keras）： 会立即进行初始化。 因为只要提供了第一层的确切输入尺寸，框架就可以根据网络架构（如各层的神经元数量、卷积核大小等）顺藤摸瓜，逐层推断出所有后续层的输入和输出形状，从而在构建模型时立即在内存中分配并初始化所有权重矩阵。依赖动态图和延后初始化的框架（如 PyTorch 的 Lazy 模块 或 早期 MXNet）： 不会立即初始化。 即使你明确指定了第一层的尺寸，只要后续层使用了延后初始化模块（例如 PyTorch 的 nn.LazyLinear），这些层依然会保持未初始化状态（UninitializedParameter）。系统需要通过执行一次真实的前向传播（Forward Pass）（哪怕是传入假数据），通过张量在计算图中的实际流动来动态推导出后续层的尺寸并完成初始化。

2. 如果指定了不匹配的维度会发生什么？会触发维度/形状不匹配错误（Shape/Dimension Mismatch Error）。深度学习的本质是一系列矩阵乘法操作。如果你手动为某一层指定了输入维度（例如 in_features=512），但上一层输出的维度是 256，那么在执行前向传播进行矩阵乘法（如 $XW + b$）时，张量的内含维度无法对齐，程序会直接报错并崩溃。

3. 如果输入具有不同的维度，需要做什么？（结合参数绑定）参数绑定（Parameter Tying / Weight Sharing）是指在网络的不同部分重用同一个层的参数（权重和偏置）。因为一个全连接层或卷积层的权重矩阵形状是固定的（例如，全连接层权重的大小为 [out_dim, in_dim]），它只能接受固定维度的输入。如果你的数据输入具有不同的维度，但你又想在后续处理中使用绑定的（共享的）参数层，你需要在进入共享层之前，先将不同维度的输入统一到相同的维度。常见的方法包括：引入独立的投影层（Projection Layers）： 为每一种不同维度的输入分配一个不共享参数的线性映射层。例如，输入 A 维度为 100，输入 B 维度为 200。你可以使用独立的 Linear(100, 512) 和 Linear(200, 512) 将它们都映射到 512 维，然后再将结果输入到参数绑定的共享层中。空间维度的聚合（全局池化）： 如果是图像或序列数据（长宽或序列长度不同，但通道数相同），可以使用全局平均池化（Global Average Pooling）或自适应池化（Adaptive Pooling）。这可以无视输入的空间分辨率，强行将特征图压缩为固定长度的向量，随后再输入到参数绑定的层。数据预处理（填充与截断）： 在输入网络之前，通过补零（Padding）将较小的维度补齐，或通过截断（Truncation）裁剪较大的维度，从源头上确保送入网络的数据维度一致。